# Week 3, day 1 (afternoon) — Worksheet 03 SOLUTIONS: the Index   (L01)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Question 7 is the one to re-read. It is a place where the lecture slide and
Pandas disagree about the answer, and the slide contradicts its own previous
slide as well.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — The Index. Run this once.
import pandas as pd

students = pd.DataFrame({
    "Name": ["Abdullah", "Sara", "Ahmed"],
    "Age": [22, 24, 21],
    "City": ["Riyadh", "Jeddah", "Dammam"],
})

# The four-row frame the lecture uses for duplicate labels.
dup = pd.DataFrame(
    {"Col1": ["D1", "D2", "D3", "D4"], "Col2": ["D1", "D2", "D3", "D4"]},
    index=[11, 12, 12, 14],
)

# Two columns that only identify a row when taken together.
pairs = pd.DataFrame({
    "Region": ["West", "West", "East", "East"],
    "Quarter": ["Q1", "Q2", "Q1", "Q2"],
    "Sales": [100, 150, 120, 180],
})

print(students)
print()
print(dup)

PART A — replacing and reading the index

### Question 1

`RangeIndex(start=0, stop=3, step=1)` -> `Index(['S1', 'S2', 'S3'], dtype='str')`.

Assigning a whole new list to `.index` is allowed and takes effect
immediately. Note the class changed too: a `RangeIndex` is a compact
description of an arithmetic sequence, not a stored list of numbers, which
is why it prints as `start/stop/step`. Give it real labels and it becomes
a plain `Index` holding them.

Hold on to the fact that this worked. Q10 does something that looks
smaller and is forbidden.

In [ ]:
print("before:", students.index)
print()
students.index = ["S1", "S2", "S3"]
print(students)
print()
print("after: ", students.index)

### Question 2

`index[0]` -> `S1`. `index[1:]` -> `Index(['S2', 'S3'], dtype='str')`. `len` -> `3`, type -> `Index`.

Reading and slicing behave exactly like a tuple, and slicing returns
another `Index` rather than a list — the type is preserved through the
operation.

The `dtype='str'` is the pandas-3 default again, the same one the L01
slide gives as `dtype="object"`.

In [ ]:
print("first label:  ", students.index[0])
print("sliced:       ", students.index[1:])
print("length:       ", len(students.index))
print("type:         ", type(students.index).__name__)

### Question 3

`loc["S2"]` -> a **Series** with `Name: S2, dtype: object`. `loc["S2", "City"]` -> `Jeddah`.

A single row comes back as a Series whose *index is the column names* —
the row has been stood on its end. Its `name` attribute is `S2`, the row's
own label, which is how it remembers where it came from.

The dtype is `object` here even though Q2 showed the columns as `str`.
That is not a contradiction: this Series holds a string, an integer and a
string together, so the only type that covers all three is `object`. Slice
a table across the rows and you get one dtype per column; slice it across
the columns and you get whatever is broad enough to hold the mixture.

In [ ]:
row = students.loc["S2"]
print(row)
print()
print("type of a single row:", type(row).__name__)
print()
print("one cell:", students.loc["S2", "City"])

PART B — duplicate labels

### Question 4

`is_unique` -> `False`. `duplicated()` -> `[False False  True False]`.

Only the *second* `12` is flagged. `duplicated()` answers 'have I seen
this label before?', reading top to bottom — so the first occurrence is
always `False`, however many copies follow.

That is the right behaviour for de-duplication, where you want to keep one
and drop the rest, and the wrong mental model if you read it as 'is this
label duplicated?'. For that question you want
`duplicated(keep=False)`, which flags every member of a repeated group.

In [ ]:
print(dup)
print()
print("is_unique:  ", dup.index.is_unique)
print("duplicated():", dup.index.duplicated())

### Question 5

`dup.loc[12]` -> a **DataFrame** of 2 rows. `dup.loc[11]` -> a **Series**, `dtype: str`.

The same expression returned two different types, decided entirely by how
many rows happened to match. This is the real cost of a duplicate label.

Code written against `dup.loc[11]` — expecting a Series, reading
`.loc[11]["Col1"]` — works perfectly and then breaks the day a second row
with that label arrives, usually from an upstream system you do not
control. It will not break loudly: `.loc[12]["Col1"]` still returns
something, just a two-element Series instead of a value.

In [ ]:
print("dup.loc[12]:")
print(dup.loc[12])
print("-> type:", type(dup.loc[12]).__name__)
print()
print("dup.loc[11]:")
print(dup.loc[11])
print("-> type:", type(dup.loc[11]).__name__)

### Question 6

`dup.loc[[12]]` -> `2` rows. `dup.loc[[11]]` -> `1` row. -> both DataFrames.

Double brackets pass a *list* of labels, and a list-of-labels selection
always returns a DataFrame — even for one label matching one row. So
`len()` means the same thing in both cases and your code stops depending
on how many duplicates the data happened to contain.

The same one-bracket/two-bracket distinction as worksheet 02 Q4, applied
to rows instead of columns. It is worth adopting as a habit: if you are
going to do anything but look at the result, use the list form.

In [ ]:
print("dup.loc[[12]] ->", len(dup.loc[[12]]), "row(s)")
print(dup.loc[[12]])
print()
print("dup.loc[[11]] ->", len(dup.loc[[11]]), "row(s)")
print(dup.loc[[11]])

# Double brackets always give a DataFrame, so len() means the same thing in
# both cases. Single brackets change shape depending on the data.

### Question 7

Result index -> `[11, 12, 12, 14, 12]`. **The slide claims `[11, 12, 13, 14, 12]`.** -> three rows now answer to label `12`.

There is no `13` anywhere in this data and there cannot be. The slide's
own previous page shows the frame being concatenated onto, and its index
is `11, 12, 12, 14`. A `13` in the result would have to come from nowhere.

So the deck contradicts itself across two consecutive pages, and the
version it prints is the one that quietly hides the problem it is trying to
teach — a table with `11,12,13,14` looks like it has a tidy unique index,
which is the opposite of the point.

What actually happens is worse and more instructive: `concat` does not
check, does not renumber, and does not warn. You now have **three** rows
under label `12`, and every `.loc[12]` from here on returns a
three-row DataFrame. If you want the tidy index the slide drew, you have
to ask for it: `pd.concat([...], ignore_index=True)`.

In [ ]:
extra = pd.DataFrame({"Col1": ["D5"], "Col2": ["D5"]}, index=[12])
result = pd.concat([dup, extra])
print(result)
print()
print("result index:", list(result.index))
print("slide claims:", [11, 12, 13, 14, 12])
print()
print("now three rows answer to label 12:")
print(result.loc[12])

PART C — building an index out of columns

### Question 8

`set_index('Region')` -> `West` and `East` each appear twice. `set_index(['Region','Quarter'])` -> a `MultiIndex`, `is_unique` **`True`**.

One column was not enough to identify a row, and Pandas let you use it
anyway — producing exactly the duplicate-label situation of Q5, this time
of your own making.

Two columns together are enough, and the result is a `MultiIndex` whose
`is_unique` is `True`. That check is the whole reason to bother: it is a
one-line proof that your chosen key actually is a key. Run it on real data
before you rely on `.loc`, because 'Region' looks like an identifier right
up until the second quarter's rows arrive.

In [ ]:
print(pairs)
print()
print("=== set_index('Region') -> labels repeat ===")
print(pairs.set_index("Region"))
print()
print("=== set_index(['Region','Quarter']) ===")
multi = pairs.set_index(["Region", "Quarter"])
print(multi)
print()
print("index type:", type(multi.index).__name__)
print("is_unique: ", multi.index.is_unique)

### Question 9

`multi.loc['West']` -> the two West quarters, indexed by `Quarter` alone. `multi.loc[('West','Q2')]` -> `Sales 150`. -> `reset_index()` puts both columns back.

Selecting on the outer level *consumes* it: the result is indexed by
`Quarter` only, because Region is no longer telling you anything — every
remaining row is West. A tuple addresses both levels at once and reaches a
single row.

`reset_index()` is the exit. It moves every index level back into ordinary
columns and restores a plain `RangeIndex`, giving you the frame you
started with. Whenever a MultiIndex is making a merge or an export awkward,
this is usually the answer.

In [ ]:
multi = pairs.set_index(["Region", "Quarter"])
print("multi.loc['West']:")
print(multi.loc["West"])
print()
print("multi.loc[('West','Q2')]:")
print(multi.loc[("West", "Q2")])
print()
print("=== reset_index() ===")
print(multi.reset_index())

### Question 10

`idx[1] = "d"` -> **raises** `TypeError: Index does not support mutable operations`.

Slicing the same object one line earlier was fine. Reading is free;
writing one label is prohibited outright.

And yet Q1 replaced the entire index with no complaint at all. That is the
asymmetry worth understanding: an index is *immutable*, but the frame's
reference to it is not. You may hand the frame a whole new index; you may
not reach into the existing one and edit it.

The reason is that indexes get shared. Two frames derived from the same
parent can point at one `Index` object, and Pandas caches lookup structures
built from it. Editing a label in place would corrupt both frames and
invalidate the cache silently. Forbidding it makes that class of bug
impossible rather than rare.

In [ ]:
idx = pd.Index(["a", "b", "c"])
print("slicing is fine:", idx[1:])

# Replacing the whole index was allowed in Q1. Editing one label is not.
idx[1] = "d"